# FAO 
## Diagnostic qualité des données _ Nettoyage des Données

**Mission :** cabinet de conseil spécialisé en politiques alimentaires - analyse mondiale de la sécurité alimentaire (données FAO 2013).

**Étape du jour :** diagnostic qualité des 5 fichiers bruts fournis, via une classe générique `DataProfiler` héritée en `ProfileurFAO`.

Étape 1 – Contrôler la qualité des données

Avant toute analyse, un diagnostic qualité a été réalisé afin d'identifier les éventuelles anomalies susceptibles d'influencer les résultats.

Diagnostic des jeux de données
Fichier	Lignes	Colonnes	Pays	Valeurs manquantes	Doublons	Types de données
Végétaux	104 871	14	175	0	0	object, int64, float64
Animaux	37 166	14	175	0	0	object, int64, float64
Population	175	14	175	174	0	object, int64
Céréales	891	14	167	0	0	object, int64
Sous-alimentation	1 020	15	204	1 435	0	object, int64, float64
Analyse des résultats
✅ Valeurs manquantes

Deux fichiers présentent des valeurs manquantes :

Population
174 valeurs manquantes, uniquement dans la colonne Symbole.
Cette colonne est descriptive et n'est pas utilisée dans les analyses.
Sous-alimentation
415 valeurs manquantes dans la colonne Valeur.
1 020 valeurs manquantes dans la colonne Note, qui est entièrement vide.
✅ Doublons

Aucun doublon n'a été détecté dans les cinq jeux de données.

➡️ Les observations sont uniques et ne nécessitent pas de suppression.

✅ Couverture géographique

Les jeux de données ne couvrent pas exactement le même nombre de pays :

Population : 175 pays
Végétaux : 175 pays
Animaux : 175 pays
Céréales : 167 pays
Sous-alimentation : 204 pays

Cette différence devra être prise en compte lors de la fusion des données.

✅ Types de données

Les types de données sont cohérents :

object : variables textuelles (pays, produits, éléments…)
int64 : codes et identifiants
float64 : valeurs quantitatives (disponibilités alimentaires, sous-alimentation…)

Aucune incohérence majeure de type n'a été observée.

🎯 Conclusion du diagnostic

Le diagnostic qualité montre que les données sont globalement de bonne qualité :

✔ Aucun doublon détecté.
✔ Types de données cohérents.
✔ Variables principales complètes dans la majorité des fichiers.
⚠ Des valeurs manquantes sont concentrées dans les fichiers Population (colonne Symbole) et Sous-alimentation (colonnes Valeur et Note).
⚠ La couverture géographique diffère selon les jeux de données (167 à 204 pays), ce qui devra être pris en compte lors des jointures.
🎤 Ce que tu peux dire à l'oral (30 à 45 secondes)

« Avant toute analyse, j'ai réalisé un diagnostic qualité sur les cinq jeux de données de la FAO. J'ai vérifié leurs dimensions, le nombre de pays couverts, les valeurs manquantes, les doublons et les types de données. Les principaux points d'attention concernent le fichier Sous-alimentation, qui contient 415 valeurs manquantes dans la colonne Valeur et une colonne Note entièrement vide, ainsi que des différences de couverture géographique entre les fichiers. Ce diagnostic m'a permis de définir une stratégie de nettoyage adaptée avant de construire le dataset global. »

1. Importation des bibliothèques

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

2. Chargement des données

In [3]:
import pandas as pd

vegetaux = pd.read_csv("../data/raw/vegetaux.csv")
animaux = pd.read_csv("../data/raw/animaux.csv")
cereales = pd.read_csv("../data/raw/cereales.csv")
population = pd.read_csv("../data/raw/population.csv")
sousalimentation = pd.read_csv("../data/raw/sousalimentation.csv")

### 1. Import et chargement des données

In [4]:
fichiers = {
    "Végétaux": vegetaux,
    "Animaux": animaux,
    "Céréales": cereales,
    "Population": population,
    "Sous-alimentation": sousalimentation
}

In [5]:
for nom, df in fichiers.items():
    print(f"{nom:20s} | {df.shape[0]:>7} lignes | {df.shape[1]} colonnes")

Végétaux             |  104871 lignes | 14 colonnes
Animaux              |   37166 lignes | 14 colonnes
Céréales             |     891 lignes | 14 colonnes
Population           |     175 lignes | 14 colonnes
Sous-alimentation    |    1020 lignes | 15 colonnes


🧹 1. Suppression des colonnes inutiles

In [6]:
# Colonnes à supprimer
colonnes_a_supprimer = [
    "Code Domaine",
    "Domaine",
    "Code Élément",
    "Code Produit",
    "Code année",
    "Symbole",
    "Description du Symbole",
    "Note"
]

# Suppression uniquement des colonnes existantes
for df in [vegetaux, animaux, population, cereales, sousalimentation]:
    df.drop(
        columns=[col for col in colonnes_a_supprimer if col in df.columns],
        inplace=True,
        errors="ignore"
    )

🧹 2. Harmonisation des noms de colonnes

In [7]:
# Renommer les colonnes pour les rendre plus lisibles
for df in [vegetaux, animaux, population, cereales, sousalimentation]:

    df.rename(columns={
        "Zone": "Zone",
        "Année": "Annee",
        "Élément": "Element",
        "Produit": "Produit",
        "Valeur": "Valeur"
    }, inplace=True)

Puis renommer les colonnes Valeur selon le fichier.

In [8]:
population.rename(columns={"Valeur":"Population"}, inplace=True)

sousalimentation.rename(
    columns={"Valeur":"Sous_alimentation"},
    inplace=True
)

🧹 3. Gestion des valeurs manquantes 
Vérification

In [9]:
print("Population")
print(population.isna().sum())

print("\nSous-alimentation")
print(sousalimentation.isna().sum())

Population
Code zone     0
Zone          0
Element       0
Produit       0
Annee         0
Unité         0
Population    0
dtype: int64

Sous-alimentation
Code zone              0
Zone                   0
Element                0
Produit                0
Annee                  0
Unité                  0
Sous_alimentation    415
dtype: int64


Suppression de la colonne Note

In [11]:
if "Note" in sousalimentation.columns:
    sousalimentation.drop(columns="Note", inplace=True)

Conservation des NaN

In [12]:
# Vérification uniquement
print(
    sousalimentation["Sous_alimentation"].isna().sum()
)

415


Les NaN sont conservés car ils correspondent à une absence de données et non à une valeur nulle.

🧹 4. Sélection de la période 2016–2018

In [13]:
sousalimentation = sousalimentation[
    sousalimentation["Annee"] == "2016-2018"
].copy()

🧹 5. Uniformisation des unités

Vérifier les unités.

In [14]:
print(vegetaux["Unité"].unique())

print(animaux["Unité"].unique())

print(population["Unité"].unique())

print(cereales["Unité"].unique())

print(sousalimentation["Unité"].unique())

['Milliers de tonnes' 'kg' 'Kcal/personne/jour' 'g/personne/jour']
['Milliers de tonnes' 'kg' 'Kcal/personne/jour' 'g/personne/jour']
['1000 personnes']
['Milliers de tonnes']
['millions']


In [15]:
vegetaux["Unité"] = vegetaux["Unité"].str.strip()

animaux["Unité"] = animaux["Unité"].str.strip()

population["Unité"] = population["Unité"].str.strip()

cereales["Unité"] = cereales["Unité"].str.strip()

sousalimentation["Unité"] = (
    sousalimentation["Unité"].str.strip()
)

🧹 6. Contrôle final

📊 Dimensions du 
Les dimensions du dataset final ont été contrôlées afin de vérifier que la fusion s'est déroulée correctement.

In [17]:
print("Population :", population.columns.tolist())
print("Végétaux :", vegetaux.columns.tolist())
print("Animaux :", animaux.columns.tolist())
print("Céréales :", cereales.columns.tolist())
print("Sous-alimentation :", sousalimentation.columns.tolist())

Population : ['Code zone', 'Zone', 'Element', 'Produit', 'Annee', 'Unité', 'Population']
Végétaux : ['Code zone', 'Zone', 'Element', 'Produit', 'Annee', 'Unité', 'Valeur']
Animaux : ['Code zone', 'Zone', 'Element', 'Produit', 'Annee', 'Unité', 'Valeur']
Céréales : ['Code zone', 'Zone', 'Element', 'Produit', 'Annee', 'Unité', 'Valeur']
Sous-alimentation : ['Code zone', 'Zone', 'Element', 'Produit', 'Annee', 'Unité', 'Sous_alimentation']


Prépare la sous-alimentation

In [18]:
print(sousalimentation["Annee"].unique())

['2016-2018']


In [19]:
sousalim_2016 = sousalimentation.copy()

In [21]:
sousalim_2016 = sousalim_2016.rename(
    columns={"Valeur": "Sous_alimentation"}
)

In [22]:
print(sousalim_2016.columns.tolist())

['Code zone', 'Zone', 'Element', 'Produit', 'Annee', 'Unité', 'Sous_alimentation']


In [23]:
print(sousalim_2016.head())
print(sousalim_2016.shape)

    Code zone            Zone Element  \
4           2     Afghanistan  Valeur   
9         202  Afrique du Sud  Valeur   
14          3         Albanie  Valeur   
19          4         Algérie  Valeur   
24         79       Allemagne  Valeur   

                                              Produit      Annee     Unité  \
4   Nombre de personnes sous-alimentées (millions)...  2016-2018  millions   
9   Nombre de personnes sous-alimentées (millions)...  2016-2018  millions   
14  Nombre de personnes sous-alimentées (millions)...  2016-2018  millions   
19  Nombre de personnes sous-alimentées (millions)...  2016-2018  millions   
24  Nombre de personnes sous-alimentées (millions)...  2016-2018  millions   

   Sous_alimentation  
4               10.6  
9                3.5  
14               0.2  
19               1.6  
24               NaN  
(204, 7)


In [24]:
dataset_global = population.merge(
    sousalim_2016,
    on="Zone",
    how="inner"
)

print(dataset_global.shape)
print(dataset_global.head())

(175, 13)
   Code zone_x            Zone          Element_x   Produit_x  Annee_x  \
0            2     Afghanistan  Population totale  Population     2013   
1          202  Afrique du Sud  Population totale  Population     2013   
2            3         Albanie  Population totale  Population     2013   
3            4         Algérie  Population totale  Population     2013   
4           79       Allemagne  Population totale  Population     2013   

          Unité_x  Population  Code zone_y Element_y  \
0  1000 personnes       30552            2    Valeur   
1  1000 personnes       52776          202    Valeur   
2  1000 personnes        3173            3    Valeur   
3  1000 personnes       39208            4    Valeur   
4  1000 personnes       82727           79    Valeur   

                                           Produit_y    Annee_y   Unité_y  \
0  Nombre de personnes sous-alimentées (millions)...  2016-2018  millions   
1  Nombre de personnes sous-alimentées (millions)...  

In [25]:
dataset_global = dataset_global[
    ["Zone", "Population", "Sous_alimentation"]
].copy()

In [26]:
print(dataset_global.head())
print(dataset_global.shape)
print(dataset_global.columns.tolist())

             Zone  Population Sous_alimentation
0     Afghanistan       30552              10.6
1  Afrique du Sud       52776               3.5
2         Albanie        3173               0.2
3         Algérie       39208               1.6
4       Allemagne       82727               NaN
(175, 3)
['Zone', 'Population', 'Sous_alimentation']


In [27]:
print("Nombre de pays :", dataset_global["Zone"].nunique())
print("Doublons :", dataset_global.duplicated().sum())

print("\nValeurs manquantes :")
print(dataset_global.isna().sum())

Nombre de pays : 175
Doublons : 0

Valeurs manquantes :
Zone                  0
Population            0
Sous_alimentation    53
dtype: int64


In [28]:
dataset_global = population.merge(
    sousalim_2016[["Zone", "Sous_alimentation"]],
    on="Zone",
    how="inner"
)

print(dataset_global.shape)
print(dataset_global.head())

(175, 8)
   Code zone            Zone            Element     Produit  Annee  \
0          2     Afghanistan  Population totale  Population   2013   
1        202  Afrique du Sud  Population totale  Population   2013   
2          3         Albanie  Population totale  Population   2013   
3          4         Algérie  Population totale  Population   2013   
4         79       Allemagne  Population totale  Population   2013   

            Unité  Population Sous_alimentation  
0  1000 personnes       30552              10.6  
1  1000 personnes       52776               3.5  
2  1000 personnes        3173               0.2  
3  1000 personnes       39208               1.6  
4  1000 personnes       82727               NaN  


Ajouter les céréales

In [29]:
datasets = {
    "Végétaux": vegetaux,
    "Animaux": animaux,
    "Population": population,
    "Céréales": cereales,
    "Sous-alimentation": sousalimentation
}

for nom, df in datasets.items():

    print("="*60)
    print(nom)

    print("Dimensions :", df.shape)

    print("\nValeurs manquantes")

    print(df.isna().sum())

    print("\nDoublons :", df.duplicated().sum())

    print("\nTypes")

    print(df.dtypes)

Végétaux
Dimensions : (104871, 7)

Valeurs manquantes
Code zone    0
Zone         0
Element      0
Produit      0
Annee        0
Unité        0
Valeur       0
dtype: int64

Doublons : 0

Types
Code zone      int64
Zone          object
Element       object
Produit       object
Annee          int64
Unité         object
Valeur       float64
dtype: object
Animaux
Dimensions : (37166, 7)

Valeurs manquantes
Code zone    0
Zone         0
Element      0
Produit      0
Annee        0
Unité        0
Valeur       0
dtype: int64

Doublons : 0

Types
Code zone      int64
Zone          object
Element       object
Produit       object
Annee          int64
Unité         object
Valeur       float64
dtype: object
Population
Dimensions : (175, 7)

Valeurs manquantes
Code zone     0
Zone          0
Element       0
Produit       0
Annee         0
Unité         0
Population    0
dtype: int64

Doublons : 0

Types
Code zone      int64
Zone          object
Element       object
Produit       object
Annee      

In [30]:
print("Dimensions :", dataset_global.shape)

print("\nPays :", dataset_global["Zone"].nunique())

print("\nDoublons :", dataset_global.duplicated().sum())

print("\nValeurs manquantes :")
print(dataset_global.isna().sum().sort_values(ascending=False))

Dimensions : (175, 8)

Pays : 175

Doublons : 0

Valeurs manquantes :
Sous_alimentation    53
Code zone             0
Zone                  0
Element               0
Produit               0
Annee                 0
Unité                 0
Population            0
dtype: int64


In [31]:
print("POPULATION")
print(population.shape)
print(population.columns.tolist())

print("\nSOUS-ALIMENTATION")
print(sousalim_2016.shape)
print(sousalim_2016.columns.tolist())

print("\nCEREALES")
print(cereales.shape)
print(cereales.columns.tolist())

print("\nVEGETAUX")
print(vegetaux.shape)
print(vegetaux.columns.tolist())

print("\nANIMAUX")
print(animaux.shape)
print(animaux.columns.tolist())

POPULATION
(175, 7)
['Code zone', 'Zone', 'Element', 'Produit', 'Annee', 'Unité', 'Population']

SOUS-ALIMENTATION
(204, 7)
['Code zone', 'Zone', 'Element', 'Produit', 'Annee', 'Unité', 'Sous_alimentation']

CEREALES
(891, 7)
['Code zone', 'Zone', 'Element', 'Produit', 'Annee', 'Unité', 'Valeur']

VEGETAUX
(104871, 7)
['Code zone', 'Zone', 'Element', 'Produit', 'Annee', 'Unité', 'Valeur']

ANIMAUX
(37166, 7)
['Code zone', 'Zone', 'Element', 'Produit', 'Annee', 'Unité', 'Valeur']


Population

In [32]:
population_clean = population[
    ["Zone", "Population"]
].copy()

Sous-alimentation

In [33]:
sousalim_clean = sousalimentation[
    ["Zone", "Sous_alimentation"]
].copy()

In [34]:
print(population_clean.shape)
print(sousalim_clean.shape)

(175, 2)
(204, 2)


unicité des pays

In [35]:
print("Population - doublons Zone :", population_clean["Zone"].duplicated().sum())
print("Sous-alimentation - doublons Zone :", sousalim_clean["Zone"].duplicated().sum())

Population - doublons Zone : 0
Sous-alimentation - doublons Zone : 0


Fusion Population + Sous-alimentation

In [36]:
dataset_global = population_clean.merge(
    sousalim_clean,
    on="Zone",
    how="inner"
)

In [37]:
print(dataset_global.shape)
print(dataset_global.head())

(175, 3)
             Zone  Population Sous_alimentation
0     Afghanistan       30552              10.6
1  Afrique du Sud       52776               3.5
2         Albanie        3173               0.2
3         Algérie       39208               1.6
4       Allemagne       82727               NaN


Contrôle de cette première fusion

In [38]:
print("Nombre de pays :", dataset_global["Zone"].nunique())
print("Doublons :", dataset_global.duplicated().sum())

print("\nValeurs manquantes :")
print(dataset_global.isna().sum())

Nombre de pays : 175
Doublons : 0

Valeurs manquantes :
Zone                  0
Population            0
Sous_alimentation    53
dtype: int64


In [39]:
print(dataset_global["Sous_alimentation"].describe())

count      122
unique      60
top       <0.1
freq        23
Name: Sous_alimentation, dtype: object


point essentiel

In [40]:
print(cereales.groupby("Zone").size().describe())

count    167.000000
mean       5.335329
std        2.408846
min        1.000000
25%        3.500000
50%        5.000000
75%        7.000000
max        9.000000
dtype: float64


In [41]:
print(vegetaux.groupby("Zone").size().describe())

count    175.000000
mean     599.262857
std      104.743033
min      266.000000
25%      537.500000
50%      619.000000
75%      676.000000
max      860.000000
dtype: float64


In [42]:
print(animaux.groupby("Zone").size().describe())

count    175.000000
mean     212.377143
std       24.363755
min       91.000000
25%      201.000000
50%      215.000000
75%      228.500000
max      258.000000
dtype: float64


In [43]:
print("=== CEREALES ===")
print(cereales["Element"].unique())
print(cereales["Produit"].unique())

print("\n=== VEGETAUX ===")
print(vegetaux["Element"].unique())
print(vegetaux["Produit"].unique())

print("\n=== ANIMAUX ===")
print(animaux["Element"].unique())
print(animaux["Produit"].unique())

=== CEREALES ===
['Production']
['Blé' 'Riz (Eq Blanchi)' 'Orge' 'Maïs' 'Millet' 'Seigle' 'Avoine'
 'Sorgho' 'Céréales, Autres']

=== VEGETAUX ===
['Production' 'Importations - Quantité' 'Variation de stock'
 'Disponibilité intérieure' 'Semences' 'Pertes' 'Nourriture'
 'Disponibilité alimentaire en quantité (kg/personne/an)'
 'Disponibilité alimentaire (Kcal/personne/jour)'
 'Disponibilité de protéines en quantité (g/personne/jour)'
 'Disponibilité de matière grasse en quantité (g/personne/jour)'
 'Aliments pour animaux' 'Exportations - Quantité' 'Traitement'
 'Autres utilisations (non alimentaire)']
['Blé' 'Riz (Eq Blanchi)' 'Orge' 'Maïs' 'Millet' 'Céréales, Autres'
 'Pommes de Terre' 'Sucre, canne' 'Sucre, betterave' 'Sucre Eq Brut'
 'Edulcorants Autres' 'Miel' 'Légumineuses Autres' 'Noix'
 'Graines de tournesol' 'Graines de coton' 'Coco (Incl Coprah)' 'Sésame'
 'Olives' 'Plantes Oleiferes, Autre' 'Huile de Soja' "Huile d'Arachide"
 'Huile de Tournesol' 'Huile de Colza&Moutarde' 'Hui

Production céréalière

In [44]:
production_cereales = (
    cereales
    .groupby("Zone", as_index=False)["Valeur"]
    .sum()
    .rename(columns={
        "Valeur": "Production_cereales"
    })
)

In [45]:
production_cereales = (
    cereales
    .groupby("Zone", as_index=False)["Valeur"]
    .sum()
    .rename(columns={
        "Valeur": "Production_cereales"
    })
)

In [46]:
print(production_cereales.head())
print(production_cereales.shape)

             Zone  Production_cereales
0     Afghanistan                 6350
1  Afrique du Sud                14155
2         Albanie                  703
3         Algérie                 4914
4       Allemagne                47757
(167, 2)


Disponibilité alimentaire en kcal

In [47]:
kcal = vegetaux[
    vegetaux["Element"] ==
    "Disponibilité alimentaire (Kcal/personne/jour)"
].copy()

In [48]:
kcal = (
    kcal
    .groupby("Zone", as_index=False)["Valeur"]
    .sum()
    .rename(columns={
        "Valeur": "Disponibilite_kcal"
    })
)

Ajouter les céréales au dataset

In [49]:
dataset_global = dataset_global.merge(
    production_cereales,
    on="Zone",
    how="inner"
)

In [50]:
dataset_global = dataset_global.merge(
    kcal,
    on="Zone",
    how="inner"
)

In [51]:
print(dataset_global.shape)
print(dataset_global.head())

(167, 5)
             Zone  Population Sous_alimentation  Production_cereales  \
0     Afghanistan       30552              10.6                 6350   
1  Afrique du Sud       52776               3.5                14155   
2         Albanie        3173               0.2                  703   
3         Algérie       39208               1.6                 4914   
4       Allemagne       82727               NaN                47757   

   Disponibilite_kcal  
0              1871.0  
1              2533.0  
2              2203.0  
3              2915.0  
4              2461.0  


In [52]:
print("Nombre de pays :", dataset_global["Zone"].nunique())

print(
    "Doublons :",
    dataset_global.duplicated(subset=["Zone"]).sum()
)

print("\nValeurs manquantes :")
print(dataset_global.isna().sum())

print("\nTypes :")
print(dataset_global.dtypes)

Nombre de pays : 167
Doublons : 0

Valeurs manquantes :
Zone                    0
Population              0
Sous_alimentation      50
Production_cereales     0
Disponibilite_kcal      0
dtype: int64

Types :
Zone                    object
Population               int64
Sous_alimentation       object
Production_cereales      int64
Disponibilite_kcal     float64
dtype: object


In [54]:
print(dataset_global.columns.tolist())

['Zone', 'Population', 'Sous_alimentation', 'Production_cereales', 'Disponibilite_kcal']


In [56]:
print(dataset_global["Sous_alimentation"].unique()[:30])

[1.060e+01 3.500e+00 2.000e-01 1.600e+00       nan 7.400e+00 2.300e+00
 2.100e+00 1.000e-01 2.420e+01 1.100e+00 1.900e+00 6.000e-01 3.000e-01
 3.800e+00 2.600e+00 2.400e+00 5.000e-01 1.214e+02 8.000e-01 4.600e+00
 4.400e+00 1.300e+00 2.160e+01 5.400e+00 1.200e+00 1.944e+02 2.200e+01
 4.000e+00 1.110e+01]


In [55]:
dataset_global["Sous_alimentation"] = pd.to_numeric(
    dataset_global["Sous_alimentation"],
    errors="coerce"
)

In [57]:
print(dataset_global["Sous_alimentation"].dtype)

print(
    "Valeurs manquantes :",
    dataset_global["Sous_alimentation"].isna().sum()
)

float64
Valeurs manquantes : 69


In [58]:
dataset_ml = dataset_global.dropna(
    subset=["Sous_alimentation"]
).copy()

In [59]:
print("Dataset avant :", dataset_global.shape)
print("Dataset ML :", dataset_ml.shape)

Dataset avant : (167, 5)
Dataset ML : (98, 5)


In [60]:
print("=" * 60)
print("DIAGNOSTIC FINAL")
print("=" * 60)

print("Nombre de lignes :", dataset_global.shape[0])
print("Nombre de colonnes :", dataset_global.shape[1])
print("Nombre de pays :", dataset_global["Zone"].nunique())

print("\nDoublons par pays :")
print(dataset_global["Zone"].duplicated().sum())

print("\nValeurs manquantes :")
print(dataset_global.isna().sum())

print("\nTypes :")
print(dataset_global.dtypes)

DIAGNOSTIC FINAL
Nombre de lignes : 167
Nombre de colonnes : 5
Nombre de pays : 167

Doublons par pays :
0

Valeurs manquantes :
Zone                    0
Population              0
Sous_alimentation      69
Production_cereales     0
Disponibilite_kcal      0
dtype: int64

Types :
Zone                    object
Population               int64
Sous_alimentation      float64
Production_cereales      int64
Disponibilite_kcal     float64
dtype: object


In [61]:
print(dataset_global["Sous_alimentation"].unique()[:50])

[1.060e+01 3.500e+00 2.000e-01 1.600e+00       nan 7.400e+00 2.300e+00
 2.100e+00 1.000e-01 2.420e+01 1.100e+00 1.900e+00 6.000e-01 3.000e-01
 3.800e+00 2.600e+00 2.400e+00 5.000e-01 1.214e+02 8.000e-01 4.600e+00
 4.400e+00 1.300e+00 2.160e+01 5.400e+00 1.200e+00 1.944e+02 2.200e+01
 4.000e+00 1.110e+01 1.460e+01 4.000e-01 7.000e-01 1.800e+00 1.140e+01
 3.300e+00 4.700e+00 8.300e+00 5.700e+00 2.500e+00 3.600e+00 2.560e+01
 1.760e+01 2.000e+00 4.000e+01 3.100e+00 1.390e+01 2.800e+00 1.000e+00
 1.220e+01]


In [62]:
print(dataset_global["Sous_alimentation"].dtype)
print("Valeurs manquantes :", dataset_global["Sous_alimentation"].isna().sum())

float64
Valeurs manquantes : 69


problématique Machine Learning claire :

« Peut-on prédire le nombre de personnes sous-alimentées dans un pays à partir de sa population, de sa production céréalière et de la disponibilité alimentaire ? »

In [63]:
dataset_ml = dataset_global.dropna(
    subset=["Sous_alimentation"]
).copy()

In [64]:
print("Dataset global :", dataset_global.shape)
print("Dataset ML :", dataset_ml.shape)
print("Valeurs manquantes cible :", 
      dataset_ml["Sous_alimentation"].isna().sum())

Dataset global : (167, 5)
Dataset ML : (98, 5)
Valeurs manquantes cible : 0


Vérification des unités

In [66]:
print("POPULATION")
print(population[["Element", "Unité"]].drop_duplicates())

print("\nSOUS-ALIMENTATION")
print(sousalimentation[["Element", "Unité"]].drop_duplicates())

print("\nCEREALES")
print(cereales[["Element", "Unité"]].drop_duplicates())

print("\nVEGETAUX - KCAL")
print(
    vegetaux[
        vegetaux["Element"] ==
        "Disponibilité alimentaire (Kcal/personne/jour)"
    ][["Element", "Unité"]].drop_duplicates()
)

POPULATION
             Element           Unité
0  Population totale  1000 personnes

SOUS-ALIMENTATION
  Element     Unité
4  Valeur  millions

CEREALES
      Element               Unité
0  Production  Milliers de tonnes

VEGETAUX - KCAL
                                          Element               Unité
8  Disponibilité alimentaire (Kcal/personne/jour)  Kcal/personne/jour


In [67]:
dataset_global["Population_millions"] = (
    dataset_global["Population"] / 1000
)

In [68]:
dataset_global.drop(
    columns=["Population"],
    inplace=True
)

In [69]:
dataset_global.rename(
    columns={"Population_millions": "Population"},
    inplace=True
)

# Pays Exclus

In [70]:
pays_population = set(population["Zone"].unique())
pays_final = set(dataset_global["Zone"].unique())

pays_exclus = sorted(
    pays_population - pays_final
)

print("Nombre de pays exclus :", len(pays_exclus))

print("\nPays exclus :")
for pays in pays_exclus:
    print("-", pays)

Nombre de pays exclus : 8

Pays exclus :
- Bermudes
- Chine
- Chine - RAS de Macao
- Islande
- Kiribati
- Polynésie française
- Saint-Kitts-et-Nevis
- Samoa


Identifier la cause de l'exclusion

In [71]:
for pays in pays_exclus:
    
    print("\n" + "=" * 50)
    print(pays)
    
    print(
        "Population :",
        pays in set(population["Zone"])
    )
    
    print(
        "Sous-alimentation :",
        pays in set(sousalimentation["Zone"])
    )
    
    print(
        "Céréales :",
        pays in set(cereales["Zone"])
    )
    
    print(
        "Végétaux :",
        pays in set(vegetaux["Zone"])
    )
    
    print(
        "Animaux :",
        pays in set(animaux["Zone"])
    )


Bermudes
Population : True
Sous-alimentation : True
Céréales : False
Végétaux : True
Animaux : True

Chine
Population : True
Sous-alimentation : True
Céréales : False
Végétaux : True
Animaux : True

Chine - RAS de Macao
Population : True
Sous-alimentation : True
Céréales : False
Végétaux : True
Animaux : True

Islande
Population : True
Sous-alimentation : True
Céréales : False
Végétaux : True
Animaux : True

Kiribati
Population : True
Sous-alimentation : True
Céréales : False
Végétaux : True
Animaux : True

Polynésie française
Population : True
Sous-alimentation : True
Céréales : False
Végétaux : True
Animaux : True

Saint-Kitts-et-Nevis
Population : True
Sous-alimentation : True
Céréales : False
Végétaux : True
Animaux : True

Samoa
Population : True
Sous-alimentation : True
Céréales : False
Végétaux : True
Animaux : True


In [72]:
pays_population = set(population["Zone"])
pays_sousalim = set(sousalimentation["Zone"])
pays_cereales = set(cereales["Zone"])
pays_vegetaux = set(vegetaux["Zone"])
pays_animaux = set(animaux["Zone"])

pays_exclus = sorted(
    pays_population - set(dataset_global["Zone"])
)

diagnostic_exclus = []

for pays in pays_exclus:
    
    absences = []
    
    if pays not in pays_sousalim:
        absences.append("Sous-alimentation")
        
    if pays not in pays_cereales:
        absences.append("Céréales")
        
    if pays not in pays_vegetaux:
        absences.append("Végétaux")
        
    if pays not in pays_animaux:
        absences.append("Animaux")
    
    diagnostic_exclus.append({
        "Pays": pays,
        "Fichiers_absents": ", ".join(absences)
    })

diagnostic_exclus = pd.DataFrame(diagnostic_exclus)

diagnostic_exclus

,Pays,Fichiers_absents
0,Bermudes,Céréales
1,Chine,Céréales
2,Chine - RAS de Macao,Céréales
3,Islande,Céréales
4,Kiribati,Céréales
5,Polynésie française,Céréales
6,Saint-Kitts-et-Nevis,Céréales
7,Samoa,Céréales


NETTOYAGE FINAL

In [74]:
# Vérifier les colonnes présentes
print(dataset_global.columns.tolist())

['Zone', 'Sous_alimentation', 'Production_cereales', 'Disponibilite_kcal', 'Population']


In [78]:
dataset_global["Population"] = (
    dataset_global["Population"] / 1000
)

In [79]:
dataset_global["Population"] = pd.to_numeric(
    dataset_global["Population"],
    errors="coerce"
)

dataset_global["Sous_alimentation"] = pd.to_numeric(
    dataset_global["Sous_alimentation"],
    errors="coerce"
)

dataset_global["Production_cereales"] = pd.to_numeric(
    dataset_global["Production_cereales"],
    errors="coerce"
)

dataset_global["Disponibilite_kcal"] = pd.to_numeric(
    dataset_global["Disponibilite_kcal"],
    errors="coerce"
)

In [80]:
print(
    "Doublons sur les pays :",
    dataset_global["Zone"].duplicated().sum()
)

Doublons sur les pays : 0


In [81]:
print(dataset_global.isna().sum())

Zone                    0
Sous_alimentation      69
Production_cereales     0
Disponibilite_kcal      0
Population              0
dtype: int64


In [82]:
dataset_ml = dataset_global.dropna(
    subset=["Sous_alimentation"]
).copy()

In [83]:
print(dataset_global.describe())

       Sous_alimentation  Production_cereales  Disponibilite_kcal  \
count          98.000000           167.000000          167.000000   
mean            7.662245         15109.850299         2301.395210   
std            23.389757         54825.138401          290.327712   
min             0.100000             0.000000         1635.000000   
25%             0.525000           216.500000         2144.500000   
50%             1.900000          2068.000000         2292.000000   
75%             5.550000          7135.500000         2449.000000   
max           194.400000        485073.000000         3188.000000   

         Population  
count  1.670000e+02  
mean   4.189067e-05  
std    1.484219e-04  
min    7.200000e-08  
25%    3.271000e-06  
50%    9.955000e-06  
75%    2.932550e-05  
max    1.385567e-03  


In [84]:
print(
    dataset_global[
        [
            "Population",
            "Sous_alimentation",
            "Production_cereales",
            "Disponibilite_kcal"
        ]
    ].describe()
)

         Population  Sous_alimentation  Production_cereales  \
count  1.670000e+02          98.000000           167.000000   
mean   4.189067e-05           7.662245         15109.850299   
std    1.484219e-04          23.389757         54825.138401   
min    7.200000e-08           0.100000             0.000000   
25%    3.271000e-06           0.525000           216.500000   
50%    9.955000e-06           1.900000          2068.000000   
75%    2.932550e-05           5.550000          7135.500000   
max    1.385567e-03         194.400000        485073.000000   

       Disponibilite_kcal  
count          167.000000  
mean          2301.395210  
std            290.327712  
min           1635.000000  
25%           2144.500000  
50%           2292.000000  
75%           2449.000000  
max           3188.000000  


In [85]:
colonnes_numeriques = [
    "Population",
    "Sous_alimentation",
    "Production_cereales",
    "Disponibilite_kcal"
]

for colonne in colonnes_numeriques:
    print(
        colonne,
        "→ valeurs négatives :",
        (dataset_global[colonne] < 0).sum()
    )

Population → valeurs négatives : 0
Sous_alimentation → valeurs négatives : 0
Production_cereales → valeurs négatives : 0
Disponibilite_kcal → valeurs négatives : 0


In [86]:
print("=" * 60)
print("CONTRÔLE QUALITÉ FINAL")
print("=" * 60)

print(f"Nombre de lignes : {dataset_global.shape[0]}")
print(f"Nombre de colonnes : {dataset_global.shape[1]}")
print(f"Nombre de pays : {dataset_global['Zone'].nunique()}")

print("\nDoublons par pays :")
print(dataset_global["Zone"].duplicated().sum())

print("\nValeurs manquantes :")
print(dataset_global.isna().sum())

print("\nTypes de données :")
print(dataset_global.dtypes)

print("\nColonnes finales :")
print(dataset_global.columns.tolist())

CONTRÔLE QUALITÉ FINAL
Nombre de lignes : 167
Nombre de colonnes : 5
Nombre de pays : 167

Doublons par pays :
0

Valeurs manquantes :
Zone                    0
Sous_alimentation      69
Production_cereales     0
Disponibilite_kcal      0
Population              0
dtype: int64

Types de données :
Zone                    object
Sous_alimentation      float64
Production_cereales      int64
Disponibilite_kcal     float64
Population             float64
dtype: object

Colonnes finales :
['Zone', 'Sous_alimentation', 'Production_cereales', 'Disponibilite_kcal', 'Population']


In [87]:
dataset_final = dataset_global.copy()

In [88]:
print(dataset_final.head())

             Zone  Sous_alimentation  Production_cereales  Disponibilite_kcal  \
0     Afghanistan               10.6                 6350              1871.0   
1  Afrique du Sud                3.5                14155              2533.0   
2         Albanie                0.2                  703              2203.0   
3         Algérie                1.6                 4914              2915.0   
4       Allemagne                NaN                47757              2461.0   

   Population  
0    0.000031  
1    0.000053  
2    0.000003  
3    0.000039  
4    0.000083  


In [89]:
dataset_global["Population"] = (
    dataset_global["Population"] * 1_000_000
)

In [90]:
print(dataset_global[[
    "Zone",
    "Population",
    "Sous_alimentation",
    "Production_cereales",
    "Disponibilite_kcal"
]].head())

             Zone  Population  Sous_alimentation  Production_cereales  \
0     Afghanistan      30.552               10.6                 6350   
1  Afrique du Sud      52.776                3.5                14155   
2         Albanie       3.173                0.2                  703   
3         Algérie      39.208                1.6                 4914   
4       Allemagne      82.727                NaN                47757   

   Disponibilite_kcal  
0              1871.0  
1              2533.0  
2              2203.0  
3              2915.0  
4              2461.0  


In [91]:
print(dataset_global["Population"].describe())

count     167.000000
mean       41.890671
std       148.421867
min         0.072000
25%         3.271000
50%         9.955000
75%        29.325500
max      1385.567000
Name: Population, dtype: float64


In [92]:
print(dataset_global[[
    "Zone",
    "Population",
    "Sous_alimentation",
    "Production_cereales",
    "Disponibilite_kcal"
]].head())

             Zone  Population  Sous_alimentation  Production_cereales  \
0     Afghanistan      30.552               10.6                 6350   
1  Afrique du Sud      52.776                3.5                14155   
2         Albanie       3.173                0.2                  703   
3         Algérie      39.208                1.6                 4914   
4       Allemagne      82.727                NaN                47757   

   Disponibilite_kcal  
0              1871.0  
1              2533.0  
2              2203.0  
3              2915.0  
4              2461.0  


In [93]:
print(dataset_global.dtypes)

Zone                    object
Sous_alimentation      float64
Production_cereales      int64
Disponibilite_kcal     float64
Population             float64
dtype: object


In [94]:
print(dataset_global.isna().sum())

Zone                    0
Sous_alimentation      69
Production_cereales     0
Disponibilite_kcal      0
Population              0
dtype: int64


DATASET POPULATION

In [97]:
dataset_global["Population"] = (
    dataset_global["Population"] * 1_000_000
)

In [98]:
print(dataset_global[[
    "Zone",
    "Population",
    "Sous_alimentation",
    "Production_cereales",
    "Disponibilite_kcal"
]].head())

             Zone  Population  Sous_alimentation  Production_cereales  \
0     Afghanistan  30552000.0               10.6                 6350   
1  Afrique du Sud  52776000.0                3.5                14155   
2         Albanie   3173000.0                0.2                  703   
3         Algérie  39208000.0                1.6                 4914   
4       Allemagne  82727000.0                NaN                47757   

   Disponibilite_kcal  
0              1871.0  
1              2533.0  
2              2203.0  
3              2915.0  
4              2461.0  


In [107]:
dataset_global["Population"] = (
    dataset_global["Population"] * 1_000_000
)

In [109]:
population_clean = population[
    ["Zone", "Population"]
].copy()

# La source est en "1000 personnes"
# Conversion en millions de personnes
population_clean["Population"] = (
    population_clean["Population"] / 1000
)

print(population_clean.head())

             Zone  Population
0     Afghanistan      30.552
1  Afrique du Sud      52.776
2         Albanie       3.173
3         Algérie      39.208
4       Allemagne      82.727


In [111]:
dataset_global = population_clean.merge(
    sousalim_clean,
    on="Zone",
    how="inner"
)

dataset_global = dataset_global.merge(
    production_cereales,
    on="Zone",
    how="inner"
)

dataset_global = dataset_global.merge(
    kcal,
    on="Zone",
    how="inner"
)

In [112]:
print(dataset_global.head())
print(dataset_global.shape)
print(dataset_global.columns.tolist())

             Zone  Population Sous_alimentation  Production_cereales  \
0     Afghanistan      30.552              10.6                 6350   
1  Afrique du Sud      52.776               3.5                14155   
2         Albanie       3.173               0.2                  703   
3         Algérie      39.208               1.6                 4914   
4       Allemagne      82.727               NaN                47757   

   Disponibilite_kcal  
0              1871.0  
1              2533.0  
2              2203.0  
3              2915.0  
4              2461.0  
(167, 5)
['Zone', 'Population', 'Sous_alimentation', 'Production_cereales', 'Disponibilite_kcal']


In [117]:
print("=" * 60)
print("CONTRÔLE QUALITÉ FINAL")
print("=" * 60)

print("Dimensions :", dataset_global.shape)
print("Nombre de pays :", dataset_global["Zone"].nunique())
print("Doublons :", dataset_global["Zone"].duplicated().sum())

print("\nValeurs manquantes :")
print(dataset_global.isna().sum())

print("\nTypes de données :")
print(dataset_global.dtypes)

print("\nStatistiques descriptives :")
print(dataset_global.describe())

CONTRÔLE QUALITÉ FINAL
Dimensions : (167, 5)
Nombre de pays : 167
Doublons : 0

Valeurs manquantes :
Zone                    0
Population              0
Sous_alimentation      50
Production_cereales     0
Disponibilite_kcal      0
dtype: int64

Types de données :
Zone                    object
Population             float64
Sous_alimentation       object
Production_cereales      int64
Disponibilite_kcal     float64
dtype: object

Statistiques descriptives :
        Population  Production_cereales  Disponibilite_kcal
count   167.000000           167.000000          167.000000
mean     41.890671         15109.850299         2301.395210
std     148.421867         54825.138401          290.327712
min       0.072000             0.000000         1635.000000
25%       3.271000           216.500000         2144.500000
50%       9.955000          2068.000000         2292.000000
75%      29.325500          7135.500000         2449.000000
max    1385.567000        485073.000000         3188.00000

In [118]:
dataset_global["Sous_alimentation"] = pd.to_numeric(
    dataset_global["Sous_alimentation"],
    errors="coerce"
)

In [119]:
print(dataset_global.dtypes)

Zone                    object
Population             float64
Sous_alimentation      float64
Production_cereales      int64
Disponibilite_kcal     float64
dtype: object


Dataset sou-alimentation

In [131]:
# Repartir de la source originale
sousalim_clean = sousalimentation[
    ["Zone", "Sous_alimentation"]
].copy()

print(sousalim_clean.dtypes)
print(sousalim_clean["Sous_alimentation"].head())

Zone                 object
Sous_alimentation    object
dtype: object
4     10.6
9      3.5
14     0.2
19     1.6
24     NaN
Name: Sous_alimentation, dtype: object


In [133]:
print(
    sousalim_clean[
        sousalim_clean["Sous_alimentation"].astype(str).str.contains(
            "<0.1",
            na=False
        )
    ]
)

                                Zone Sous_alimentation
89                           Barbade              <0.1
104                           Belize              <0.1
144                Brunéi Darussalam              <0.1
164                       Cabo Verde              <0.1
199             Chine - RAS de Macao              <0.1
214                           Chypre              <0.1
264                        Dominique              <0.1
299                          Estonie              <0.1
324                            Fidji              <0.1
394                           Guyana              <0.1
424                     Îles Salomon              <0.1
499                         Kiribati              <0.1
544                Macédoine du Nord              <0.1
564                         Maldives              <0.1
584                          Maurice              <0.1
664               Nouvelle-Calédonie              <0.1
739              Polynésie française              <0.1
829  Saint

In [134]:
sousalim_clean["Sous_alimentation_inf_0_1"] = (
    sousalim_clean["Sous_alimentation"]
    .astype(str)
    .str.strip()
    .eq("<0.1")
)

print(
    "Nombre de valeurs <0.1 :",
    sousalim_clean["Sous_alimentation_inf_0_1"].sum()
)

Nombre de valeurs <0.1 : 23


In [139]:
sousalim_clean["Sous_alimentation"] = (
    sousalim_clean["Sous_alimentation"]
    .astype(str)
    .str.strip()
    .str.replace("<0.1", "0.05", regex=False)
)

sousalim_clean["Sous_alimentation"] = pd.to_numeric(
    sousalim_clean["Sous_alimentation"],
    errors="coerce"
)

In [142]:
print(sousalim_clean.dtypes)

print("\nValeurs manquantes :")
print(sousalim_clean["Sous_alimentation"].isna().sum())

print("\nValeurs estimées <0.1 :")
print(sousalim_clean["Sous_alimentation_inf_0_1"].sum())

Zone                          object
Sous_alimentation            float64
Sous_alimentation_inf_0_1       bool
dtype: object

Valeurs manquantes :
82

Valeurs estimées <0.1 :
23


In [143]:
print(
    sousalim_clean[
        sousalim_clean["Sous_alimentation_inf_0_1"]
    ][[
        "Zone",
        "Sous_alimentation",
        "Sous_alimentation_inf_0_1"
    ]].head(10)
)

                     Zone  Sous_alimentation  Sous_alimentation_inf_0_1
89                Barbade               0.05                       True
104                Belize               0.05                       True
144     Brunéi Darussalam               0.05                       True
164            Cabo Verde               0.05                       True
199  Chine - RAS de Macao               0.05                       True
214                Chypre               0.05                       True
264             Dominique               0.05                       True
299               Estonie               0.05                       True
324                 Fidji               0.05                       True
394                Guyana               0.05                       True


In [144]:
print("Valeurs <0.1 :", 
      sousalim_clean["Sous_alimentation_inf_0_1"].sum())

print("Valeurs réellement manquantes :",
      sousalim_clean["Sous_alimentation"].isna().sum())

Valeurs <0.1 : 23
Valeurs réellement manquantes : 82


In [146]:
dataset_global = population_clean.merge(
    sousalim_clean,
    on="Zone",
    how="inner"
)

dataset_global = dataset_global.merge(
    production_cereales,
    on="Zone",
    how="inner"
)

dataset_global = dataset_global.merge(
    kcal,
    on="Zone",
    how="inner"
)

In [147]:
print(dataset_global.shape)
print(dataset_global.columns.tolist())

(167, 6)
['Zone', 'Population', 'Sous_alimentation', 'Sous_alimentation_inf_0_1', 'Production_cereales', 'Disponibilite_kcal']


In [148]:
dataset_global

,Zone,Population,Sous_alimentation,Sous_alimentation_inf_0_1,Production_cereales,Disponibilite_kcal
0,Afghanistan,30.552,10.6,False,6350,1871.0
1,Afrique du Sud,52.776,3.5,False,14155,2533.0
2,Albanie,3.173,0.2,False,703,2203.0
3,Algérie,39.208,1.6,False,4914,2915.0
4,Allemagne,82.727,NaN,False,47757,2461.0
...,...,...,...,...,...,...
162,Venezuela (République bolivarienne du),30.405,6.8,False,3029,2157.0
163,Viet Nam,91.680,8.8,False,34567,2169.0
164,Yémen,24.407,11.0,False,863,2028.0
165,Zambie,14.539,8.0,False,2886,1818.0


In [149]:
dataset_final = dataset_global.drop(
    columns=["Sous_alimentation_inf_0_1"]
).copy()

In [150]:
print(dataset_final.shape)
print(dataset_final.columns.tolist())

(167, 5)
['Zone', 'Population', 'Sous_alimentation', 'Production_cereales', 'Disponibilite_kcal']


In [151]:
print("=" * 60)
print("DATASET FINAL")
print("=" * 60)

print("Dimensions :", dataset_final.shape)
print("Pays :", dataset_final["Zone"].nunique())
print("Doublons :", dataset_final["Zone"].duplicated().sum())

print("\nValeurs manquantes :")
print(dataset_final.isna().sum())

print("\nTypes :")
print(dataset_final.dtypes)

DATASET FINAL
Dimensions : (167, 5)
Pays : 167
Doublons : 0

Valeurs manquantes :
Zone                    0
Population              0
Sous_alimentation      50
Production_cereales     0
Disponibilite_kcal      0
dtype: int64

Types :
Zone                    object
Population             float64
Sous_alimentation      float64
Production_cereales      int64
Disponibilite_kcal     float64
dtype: object


In [152]:
import os

os.makedirs("../data/processed", exist_ok=True)

dataset_final.to_csv(
    "../data/processed/dataset_final.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✅ Dataset final enregistré.")

✅ Dataset final enregistré.


« Le dataset final contient 167 pays et cinq variables analytiques. Chaque ligne correspond à un pays. J'ai contrôlé les doublons, les types, les valeurs manquantes et les unités. Les valeurs <0,1 de sous-alimentation ont été identifiées séparément lors du nettoyage, tandis que les données réellement manquantes ont été conservées. Le dataset est maintenant prêt pour l'analyse exploratoire. »

À ce stade, je considère le nettoyage terminé. ✅

« Après les différentes étapes de nettoyage et de fusion, le dataset final couvre 167 pays. Le contrôle qualité a confirmé l'absence de doublons. J'ai identifié 50 valeurs manquantes dans la variable de sous-alimentation. Ces valeurs n'ont pas été remplacées par zéro, car une absence de donnée ne signifie pas une absence de sous-alimentation. Elles seront donc exclues uniquement de la phase d'apprentissage du modèle, puisque la variable cible doit être connue pour entraîner une régression. »

« Le contrôle final montre que la variable de sous-alimentation est bien numérique et exprimée en millions de personnes. 50 valeurs sont manquantes sur les 167 pays conservés après les jointures. Ces valeurs n'ont pas été remplacées artificiellement, car une donnée manquante ne signifie pas une absence de sous-alimentation. Les observations dont la variable cible est inconnue seront exclues uniquement de l'apprentissage du modèle. »

Justification de la jointure et des exclusions

Une jointure interne (INNER JOIN) a été retenue afin de conserver uniquement les pays disposant des variables nécessaires à la construction du dataset final.

La comparaison des couvertures géographiques a permis d'identifier 8 pays ou territoires exclus : Bermudes, Chine, Chine - RAS de Macao, Islande, Kiribati, Polynésie française, Saint-Kitts-et-Nevis et Samoa.

Dans les huit cas, l'absence est liée au fichier Céréales. Ces territoires sont donc exclus du dataset final afin de maintenir une structure homogène et d'éviter de créer artificiellement des valeurs manquantes.

Cette absence ne permet cependant pas de conclure à une absence de production céréalière. L'hypothèse retenue est que ces données ne sont pas disponibles ou ne sont pas couvertes dans le fichier Céréales utilisé. Cette hypothèse est donc distinguée d'un fait directement observé dans les données.

problématique Machine Learning claire :

« Peut-on prédire le nombre de personnes sous-alimentées dans un pays à partir de sa population, de sa production céréalière et de la disponibilité alimentaire ? »

# Constat

Lors du diagnostic qualité, une différence de structure temporelle a été identifiée entre les jeux de données :

Les fichiers Population, Végétaux, Animaux et Céréales sont organisés par année (ex. : 2013).
Le fichier Sous-alimentation est organisé par périodes glissantes de trois ans (ex. : 2012–2014, 2013–2015, …, 2016–2018).

Cette différence empêche une fusion directe sur la colonne Année, car les clés ne correspondent pas.

# Décision retenue

Pour garantir la cohérence des données, une seule période de référence a été conservée :

2016–2018

Cette période est la plus récente disponible dans le fichier Sous-alimentation et constitue une référence stable pour comparer les indicateurs entre les pays.

La jointure est ensuite réalisée sur la variable Zone, commune à l'ensemble des jeux de données.

# Pourquoi ce choix ?

Ce choix permet de :

assurer la cohérence entre les différentes sources de données ;
éviter les erreurs de fusion liées à des formats temporels incompatibles ;
conserver une seule valeur de sous-alimentation par pays ;
construire un dataset adapté à l'analyse exploratoire et à la modélisation.